# Last public candidate: Section 5's **raw** dump

Section 3 is ruled out as LLMGPR's Foursquare source by a hard bound: it contains
**429 distinct categories in the entire global dump**, and LLMGPR reports **436** — for
NYC alone *and* for three cities, identically, in both versions of the paper. You cannot
observe 436 categories in a vocabulary of 429.

Section 5's *filtered* file is ruled out too: 592,341 check-ins in the three cities against
a floor of 809,620 implied by their own table.

That leaves one public candidate: Section 5's **raw** files, which the earlier notebook
deliberately skipped —

| | filtered (`dataset_WWW_*`) | raw (`raw_*`) |
|---|---|---|
| check-ins | 22,809,624 | **90,048,627** |
| users | 114,324 | **2,733,324** |
| venues | 3,820,891 | **11,180,160** |

**The decisive test runs first and costs about ten minutes.** `raw_POIs.txt` carries its own
category vocabulary. If it holds fewer than 436 categories, no public Yang dump can be their
source and we stop. If it holds 436 or more — and especially if the three cities alone give
exactly 436 — this is very likely it, and the notebook continues to the full statistics.

Then, per the convention we settled on, the statistics table reports **POIs with >=1 check-in**
(their column, as measured — their #POIs is provably not a post-filter count), with the
>=10-core numbers printed alongside for honesty.

Disk peaks around 8.9 GB of Kaggle's 20 GB; the zip is deleted after extraction.
Runtime roughly 30-50 min.

## 0. Setup

In [1]:
import os, sys, re, zipfile, subprocess, gc
import pandas as pd, numpy as np

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-4000:])
    if p.stderr: print(p.stderr[-4000:], file=sys.stderr)
    if check and p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p

def du(): sh("df -h /kaggle/working | tail -1", check=False)

def save_table(df, stem):
    # parquet when an engine is available, csv otherwise
    try:
        df.to_parquet(f"{stem}.parquet", index=False); out = f"{stem}.parquet"
    except ImportError:
        df.to_csv(f"{stem}.csv", index=False); out = f"{stem}.csv"
    print("wrote", out)
    return out

def find(pat, root=WORK):
    hits = []
    for dp, _, fns in os.walk(root):
        if "__MACOSX" in dp: continue
        for fn in fns:
            if re.search(pat, fn, re.I) and not fn.startswith("._"):
                hits.append(os.path.join(dp, fn))
    return sorted(hits)

def fetch_drive(file_id, dest, expected_bytes=None, resourcekey=None):
    # Download a public Drive file and verify it. gdown's fuzzy parser normalises the
    # link to uc?id=... and DROPS the resourcekey that legacy ids require -> 403 HTML.
    # drive.usercontent.google.com honours resourcekey + confirm=t.
    if os.path.exists(dest) and (expected_bytes is None
                                 or os.path.getsize(dest) == expected_bytes):
        print(f"already have {dest} ({os.path.getsize(dest)/1024**3:.2f} GB)")
    else:
        url = (f"https://drive.usercontent.google.com/download?id={file_id}"
               f"&export=download&confirm=t"
               + (f"&resourcekey={resourcekey}" if resourcekey else ""))
        print("$ curl", url, flush=True)
        rc = subprocess.run(f'curl -L --fail --retry 3 --retry-delay 5 -o "{dest}" "{url}"',
                            shell=True).returncode
        if rc: raise RuntimeError(f"curl failed (exit {rc})")
    with open(dest, "rb") as f: magic = f.read(4)
    assert magic == b"PK\x03\x04", (
        f"{dest} is not a zip (starts {magic!r}) - Drive served an HTML page.")
    with zipfile.ZipFile(dest) as z:
        names = [n for n in z.namelist() if not n.endswith("/") and "__MACOSX" not in n]
    print(f"OK {dest}: {os.path.getsize(dest)/1024**3:.2f} GB, {len(names)} entries")
    return dest

# ---- LLMGPR targets -----------------------------------------------------------
TARGET_3CITY = dict(users=7_507, groups=1_715, pois=80_962, cats=436,
                    checkins=1_214_631, group_checkins=12_594)   # CIKM'25
TARGET_NYC   = dict(users=6_078, groups=1_557, pois=63_445, cats=436,
                    checkins=923_856, group_checkins=10_899)     # arXiv v1
MIN_INTERACTIONS = 10
CHUNK = 2_000_000

# Bounding boxes, plus Yang's own city centres (read from dataset_TIST2015_Cities.txt in
# the previous run) so this notebook does not need the Section-3 zip.
CITY_BBOX = {
    "New York":    dict(lon_min=-74.3,  lon_max=-73.6,  lat_min=40.4, lat_max=41.0),
    "Chicago":     dict(lon_min=-88.0,  lon_max=-87.5,  lat_min=41.6, lat_max=42.1),
    "Los Angeles": dict(lon_min=-118.7, lon_max=-117.6, lat_min=33.6, lat_max=34.4),
}
CITY_CENTRE = {"New York": (40.707864, -73.905237),
               "Chicago":  (41.826546, -87.641298),
               "Los Angeles": (34.000002, -118.250001)}
CENTRE_RADIUS_KM = 25      # reproduced Section-3 NYC POIs to within 1.9%
print("setup ok")

setup ok


## 1. Fetch Section 5 and extract the **raw** members

In [2]:
WWW_ZIP = f"{WORK}/dataset_WWW2019.zip"
RAW_POIS = find(r"raw_POIs\.txt$")
RAW_CK   = find(r"raw_Checkins.*\.txt$")

if not (RAW_POIS and RAW_CK):
    fetch_drive("1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8", WWW_ZIP, expected_bytes=2_684_000_558)
    with zipfile.ZipFile(WWW_ZIP) as z:
        wanted = [n for n in z.namelist()
                  if "__MACOSX" not in n and not n.endswith("/")
                  and re.search(r"raw_(POIs|Checkins)", n)]
        print("extracting (5.7 GB + 0.7 GB, this takes a few minutes):", wanted)
        for n in wanted:
            z.extract(n, WORK); print("  done", n, flush=True)
    os.remove(WWW_ZIP)
    RAW_POIS = find(r"raw_POIs\.txt$"); RAW_CK = find(r"raw_Checkins.*\.txt$")

RAW_POIS, RAW_CK = RAW_POIS[0], RAW_CK[0]
print("raw POIs     :", RAW_POIS, f"{os.path.getsize(RAW_POIS)/1024**3:.2f} GB")
print("raw check-ins:", RAW_CK,   f"{os.path.getsize(RAW_CK)/1024**3:.2f} GB")
du()

$ curl https://drive.usercontent.google.com/download?id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&export=download&confirm=t


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2559M  100 2559M    0     0   135M      0  0:00:18  0:00:18 --:--:-- 96.1M


OK /kaggle/working/dataset_WWW2019.zip: 2.50 GB, 6 entries
extracting (5.7 GB + 0.7 GB, this takes a few minutes): ['dataset_WWW2019/raw_Checkins_anonymized.txt', 'dataset_WWW2019/raw_POIs.txt']
  done dataset_WWW2019/raw_Checkins_anonymized.txt
  done dataset_WWW2019/raw_POIs.txt
raw POIs     : /kaggle/working/dataset_WWW2019/raw_POIs.txt 0.66 GB
raw check-ins: /kaggle/working/dataset_WWW2019/raw_Checkins_anonymized.txt 5.69 GB
$ df -h /kaggle/working | tail -1
/dev/loop1       20G  6.4G   14G  33% /kaggle/working



## 2. THE DECISIVE TEST — how many categories does `raw_POIs.txt` carry?

Section 3 has 429 globally. LLMGPR reports 436. If this dump is under 436 as well, no public
Yang release can be their source, and the rest of the notebook is academic.

In [3]:
POI_COLS = ["venue_id", "lat", "lon", "category", "country"]

all_cats, city_cats = set(), {c: set() for c in CITY_BBOX}
city_venues = {c: set() for c in CITY_BBOX}
ctr_venues  = {c: set() for c in CITY_BBOX}
VENUE2CAT, seen = {}, 0

for ch in pd.read_csv(RAW_POIS, sep="\t", header=None, names=POI_COLS,
                      dtype={"venue_id": str, "category": str},
                      on_bad_lines="skip", chunksize=CHUNK):
    seen += len(ch)
    all_cats |= set(ch["category"].dropna().unique())
    ch["lat"] = pd.to_numeric(ch["lat"], errors="coerce")
    ch["lon"] = pd.to_numeric(ch["lon"], errors="coerce")
    ch = ch.dropna(subset=["lat", "lon"])
    for city, b in CITY_BBOX.items():
        sub = ch[ch["lon"].between(b["lon_min"], b["lon_max"]) &
                 ch["lat"].between(b["lat_min"], b["lat_max"])]
        if len(sub):
            city_venues[city] |= set(sub["venue_id"])
            city_cats[city]   |= set(sub["category"].dropna().unique())
            VENUE2CAT.update(zip(sub["venue_id"], sub["category"]))
        clat, clon = CITY_CENTRE[city]
        d = np.hypot((ch["lat"].to_numpy() - clat) * 111.32,
                     (ch["lon"].to_numpy() - clon) * 111.32 * np.cos(np.radians(clat)))
        ctr_venues[city] |= set(ch.loc[d <= CENTRE_RADIUS_KM, "venue_id"])
    print(f"\rscanned {seen:,} POIs | global cats {len(all_cats):,}", end="", flush=True)

cats_3city = set().union(*city_cats.values())
print(f"\n\nraw_POIs.txt rows          : {seen:,}")
print(f"categories, GLOBAL         : {len(all_cats):,}")
print(f"categories, 3 cities (bbox): {len(cats_3city):,}")
print(f"LLMGPR reports             : {TARGET_3CITY['cats']:,}")
print()
for city in CITY_BBOX:
    print(f"  {city:<12} bbox {len(city_venues[city]):>8,} venues | "
          f"cats {len(city_cats[city]):>4,} | centre R={CENTRE_RADIUS_KM}km {len(ctr_venues[city]):>8,}")

if len(all_cats) < TARGET_3CITY["cats"]:
    print(f"\n*** RULED OUT: {len(all_cats)} categories < {TARGET_3CITY['cats']}. "
          "No public Yang dump can be LLMGPR's source. Stop here - adopt Section 3 "
          "plus the recovered friendships and report our own table. ***")
else:
    print(f"\n*** VIABLE: {len(all_cats)} categories >= {TARGET_3CITY['cats']}. "
          "Continue to the check-in scan below. ***")
    if len(cats_3city) == TARGET_3CITY["cats"]:
        print("*** AND the three cities give EXACTLY 436 - that is a smoking gun. ***")

scanned 11,180,160 POIs | global cats 519

raw_POIs.txt rows          : 11,180,160
categories, GLOBAL         : 519
categories, 3 cities (bbox): 436
LLMGPR reports             : 436

  New York     bbox  113,326 venues | cats  433 | centre R=25km   97,019
  Chicago      bbox   41,485 venues | cats  415 | centre R=25km   34,751
  Los Angeles  bbox   82,917 venues | cats  425 | centre R=25km   47,377

*** VIABLE: 519 categories >= 436. Continue to the check-in scan below. ***
*** AND the three cities give EXACTLY 436 - that is a smoking gun. ***


## 3. Stream the 90M raw check-ins

In [4]:
CK_COLS = ["user_id", "venue_id", "utc_time", "tz_offset"]
keep = pd.Index(sorted(set().union(*city_venues.values(), *ctr_venues.values())))
print(f"target venue set: {len(keep):,}")

parts, seen = [], 0
for ch in pd.read_csv(RAW_CK, sep="\t", header=None, names=CK_COLS,
                      dtype={"user_id": str, "venue_id": str, "utc_time": str},
                      usecols=[0, 1, 2, 3], on_bad_lines="skip", chunksize=CHUNK):
    seen += len(ch)
    parts.append(ch[ch["venue_id"].isin(keep)])
    print(f"\rscanned {seen:,}", end="", flush=True)
ck = pd.concat(parts, ignore_index=True); del parts; gc.collect()

print(f"\n\nin-scope check-ins (unfiltered): {len(ck):,}")
print(f"distinct users {ck['user_id'].nunique():,} | distinct venues {ck['venue_id'].nunique():,}")
print(f"\nfor comparison, same three boxes:")
print(f"  Section 5 filtered : 592,341 check-ins")
print(f"  Section 3          : 975,405 check-ins")
print(f"  LLMGPR target      : {TARGET_3CITY['checkins']:,}")
save_table(ck, f"{WORK}/raw_3city")

target venue set: 237,807
scanned 90,048,627

in-scope check-ins (unfiltered): 2,228,002
distinct users 152,487 | distinct venues 237,807

for comparison, same three boxes:
  Section 5 filtered : 592,341 check-ins
  Section 3          : 975,405 check-ins
  LLMGPR target      : 1,214,631
wrote /kaggle/working/raw_3city.parquet


'/kaggle/working/raw_3city.parquet'

## 4. Statistics table

Reported on **their convention** — `#POIs` = venues with >=1 check-in, since their column is
provably not a post-filter count (reproducing 63,445 NYC POIs under a >=10 filter would need
91% of NYC venues to clear ten check-ins; the measured figure on Section 3 is 17%).
The >=10-core numbers are printed underneath so the table stays honest either way.

In [5]:
def k_core(df, k=MIN_INTERACTIONS):
    while True:
        before = len(df)
        vc = df["user_id"].value_counts();  df = df[df["user_id"].isin(vc[vc >= k].index)]
        vc = df["venue_id"].value_counts(); df = df[df["venue_id"].isin(vc[vc >= k].index)]
        if len(df) == before or df.empty: return df

def report(assign, label):
    print(f"\n### {label}")
    hdr = f"{'city':<15}{'users':>9}{'POIs>=1':>10}{'cats':>7}{'check-ins':>13}{'ck/user':>9}"
    print(hdr); print("-"*len(hdr))
    per = {}
    for city, vset in assign.items():
        d = ck[ck["venue_id"].isin(vset)]
        per[city] = d
        u = max(d["user_id"].nunique(), 1)
        cats = pd.Series([VENUE2CAT.get(v) for v in d["venue_id"].unique()]).nunique()
        print(f"{city:<15}{d['user_id'].nunique():>9,}{d['venue_id'].nunique():>10,}"
              f"{cats:>7,}{len(d):>13,}{len(d)/u:>9.1f}")
    allc = pd.concat(per.values(), ignore_index=True).drop_duplicates()
    u = max(allc["user_id"].nunique(), 1)
    cats = pd.Series([VENUE2CAT.get(v) for v in allc["venue_id"].unique()]).nunique()
    print(f"{'ALL 3':<15}{allc['user_id'].nunique():>9,}{allc['venue_id'].nunique():>10,}"
          f"{cats:>7,}{len(allc):>13,}{len(allc)/u:>9.1f}")
    core = k_core(allc.copy())
    uc = max(core["user_id"].nunique(), 1)
    print(f"{'  (>=10 core)':<15}{core['user_id'].nunique():>9,}{core['venue_id'].nunique():>10,}"
          f"{'':>7}{len(core):>13,}{len(core)/uc:>9.1f}")
    for name, t in (("TARGET 3city", TARGET_3CITY), ("TARGET NYConly", TARGET_NYC)):
        print(f"{name:<15}{t['users']:>9,}{t['pois']:>10,}{t['cats']:>7,}"
              f"{t['checkins']:>13,}{t['checkins']/t['users']:>9.1f}")
    return allc, core

all_bbox, core_bbox = report(city_venues, "bounding box  (their #POIs convention: >=1 check-in)")
all_ctr,  core_ctr  = report(ctr_venues,  f"nearest centre, R={CENTRE_RADIUS_KM} km")
save_table(all_bbox,  f"{WORK}/raw_3city_ge1")
save_table(core_bbox, f"{WORK}/raw_3city_10core")


### bounding box  (their #POIs convention: >=1 check-in)
city               users   POIs>=1   cats    check-ins  ck/user
---------------------------------------------------------------
New York          88,853   113,326    433    1,164,743     13.1
Chicago           36,467    41,485    415      405,929     11.1
Los Angeles       50,120    82,917    425      657,084     13.1
ALL 3            152,480   237,728    436    2,227,315     14.6
  (>=10 core)     30,901    37,103           1,379,959     44.7
TARGET 3city       7,507    80,962    436    1,214,631    161.8
TARGET NYConly     6,078    63,445    436      923,856    152.0

### nearest centre, R=25 km
city               users   POIs>=1   cats    check-ins  ck/user
---------------------------------------------------------------
New York          86,970    97,019    432    1,058,938     12.2
Chicago           29,450    34,751    413      343,015     11.6
Los Angeles       41,553    47,377    417      414,034     10.0
ALL 3            

'/kaggle/working/raw_3city_10core.parquet'

In [6]:
# Verdict: how close does the raw dump get, on every column at once?
print(f"{'column':<22}{'raw dump':>14}{'target':>14}{'ratio':>9}")
print("-"*59)
rows = [("users",          all_bbox["user_id"].nunique(),  TARGET_3CITY["users"]),
        ("POIs (>=1 ck)",  all_bbox["venue_id"].nunique(), TARGET_3CITY["pois"]),
        ("categories",     len(cats_3city),                TARGET_3CITY["cats"]),
        ("check-ins",      len(all_bbox),                  TARGET_3CITY["checkins"])]
for name, got, want in rows:
    print(f"{name:<22}{got:>14,}{want:>14,}{got/want:>9.2f}x")
close = sum(1 for _, g, w in rows if 0.8 <= g/w <= 1.25)
print(f"\n{close}/4 columns within +/-25% of their table.")
print("4/4  -> this is their dataset; switch the pipeline to the raw dump.")
print("<=2/4 -> no public Yang dump reproduces LLMGPR Table 1; adopt Section 3 + "
      "recovered friendships and report our own numbers.")

column                      raw dump        target    ratio
-----------------------------------------------------------
users                        152,480         7,507    20.31x
POIs (>=1 ck)                237,728        80,962     2.94x
categories                       436           436     1.00x
check-ins                  2,227,315     1,214,631     1.83x

1/4 columns within +/-25% of their table.
4/4  -> this is their dataset; switch the pipeline to the raw dump.
<=2/4 -> no public Yang dump reproduces LLMGPR Table 1; adopt Section 3 + recovered friendships and report our own numbers.


In [7]:
# Optional: reclaim ~6.4 GB once the parquets are written.
for p in (RAW_CK, RAW_POIS):
    if os.path.exists(p): os.remove(p); print("removed", p)
du()

removed /kaggle/working/dataset_WWW2019/raw_Checkins_anonymized.txt
removed /kaggle/working/dataset_WWW2019/raw_POIs.txt
$ df -h /kaggle/working | tail -1
/dev/loop1       20G  114M   20G   1% /kaggle/working



## What to send back

The category block from section 2 (**global** and **3-city** counts against 436), and the
4-column verdict table from section 4.

Whichever way it lands, the Section-3 result already stands: 97% of Section-5 friendship
users are recoverable into Section-3 id space at vote purity 1.00, so group construction is
unblocked either way. Only the check-in source is in question here.